# CODE HAVE BEN PUT OUT OF COMMISSION

# MSc Supervisor Extraction - Local First

This notebook is a modular, local-first extraction pipeline for supervisor names in MSc thesis PDFs.

## Architecture
1. `CONFIG`: one central controller for paths, keywords, regex, and confidence thresholds.
2. Data Loader: local PDF file discovery and document loading.
3. Extraction Engine: pure functions for standardization, context finding, extraction, scoring, and deduplication.
4. Execution Loop: fail-fast per file with side-by-side debug output (`raw` vs `cleaned`).

## Design Goals
- Zero redundant helper logic.
- Easy debugging and deterministic behavior.
- Future cloud migration by replacing one loader function only.

In [ ]:
### ==== IMPORTS ====
import logging
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import pypdf
import spacy

# Logging is critical for fail-fast debugging in batch mode.
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("supervisor_extraction")

# Load NLP model with graceful fallback.
try:
    nlp = spacy.load("en_core_web_sm")
    logger.info("spaCy model loaded: en_core_web_sm")
except OSError:
    logger.warning("spaCy model not found. Falling back to regex-only extraction.")
    nlp = None

INFO | spaCy model loaded: en_core_web_sm


In [33]:
### ==== CONFIG (GLOBAL CONTROLLER) ====
CONFIG: Dict[str, object] = {
    "LOCAL_DATA_DIR": "./Data/RAW_test",
    "MAX_FILES": None,  # Set an int for quick tests, e.g. 10
    "MAX_PAGES_TO_SCAN": 8,
    "EXTRACTION_KEYWORDS": [
        "supervisor",
        "supervisors",
        "main supervisor",
        "co-supervisor",
        "advisor",
        "adviser",
        "vejleder",
        "vejledere",
        "supervised by",
    ],
    "RELATED_EXCLUSION_TERMS": [
        "master thesis",
        "education",
        "ph.d.",
        "phd",
        "msc",
        "bsc",
    ],
    "NOT_NAME_PHRASES": [
        "technical university of denmark",
        "danmarks tekniske universitet",
        "department",
        "faculty",
        "institute",
        "supervised by",
        "dtu",
    ],
    "ROLE_TERMS": [
        "director",
        "department",
        "faculty",
        "institute",
        "university",
        "researcher",
        "student",
        "group",
        "laboratory",
        "lab",
        "chair",
    ],
    "REGEX_PATTERNS": {
        # Line-level extraction around labels such as "Supervisors: John Doe and Jane Doe"
        "keyword_line": r"(?i)(?:supervisors?|main\s+supervisor|co-?supervisor|advisors?|advisers?|vejledere?)\s*[:\-]\s*(.+)",
        # Person-like token sequence with Danish letters and initials.
        "name": r"\b[A-ZÆØÅ][a-zæøåA-ZÆØÅ\-']+(?:\s+[A-ZÆØÅ][a-zæøåA-ZÆØÅ\-']+){1,4}\b",
        # Candidate split delimiters.
        "split": r"\s*(?:,|\band\b|\bog\b|&)\s*",
    },
    "CONFIDENCE_THRESHOLDS": {
        "min_confidence": 0.55,
        "keyword_bonus": 0.35,
        "regex_bonus": 0.25,
        "nlp_person_bonus": 0.20,
        "length_bonus": 0.10,
        "penalty_role_term": 0.35,
    },
}

# Precompiled regex for speed and consistency.
NAME_REGEX = re.compile(CONFIG["REGEX_PATTERNS"]["name"])  # type: ignore[index]
KEYWORD_LINE_REGEX = re.compile(CONFIG["REGEX_PATTERNS"]["keyword_line"])  # type: ignore[index]
SPLIT_REGEX = re.compile(CONFIG["REGEX_PATTERNS"]["split"], flags=re.IGNORECASE)  # type: ignore[index]

In [34]:
### ==== DATA LOADER (LOCAL ONLY) ====
def locate_project_root(start: Path) -> Path:
    """Find project root by looking for repo marker files."""
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "pyproject.toml").exists() or (candidate / "README.md").exists():
            return candidate
    return current


def resolve_local_data_dir(local_dir: str) -> Path:
    """
    Resolve local data directory robustly.
    Priority:
    1) Absolute path as-is
    2) Relative to current working directory
    3) Relative to project root
    """
    candidate = Path(local_dir)
    if candidate.is_absolute() and candidate.exists():
        return candidate

    cwd_candidate = (Path.cwd() / candidate).resolve()
    if cwd_candidate.exists():
        return cwd_candidate

    root = locate_project_root(Path.cwd())
    root_candidate = (root / candidate).resolve()
    if root_candidate.exists():
        return root_candidate

    raise FileNotFoundError(
        f"Local data directory not found. Tried: {cwd_candidate} and {root_candidate}"
    )


def list_local_pdf_files(local_dir: str, max_files: Optional[int] = None) -> List[Path]:
    """Single loader function: replace this one function for future GCP integration."""
    base = resolve_local_data_dir(local_dir)
    pdf_files = sorted(base.glob("*.pdf"))

    if max_files is not None:
        pdf_files = pdf_files[:max_files]

    logger.info("PDF discovery complete | directory=%s | files=%d", base, len(pdf_files))
    return pdf_files


def load_pdf_pages(pdf_path: Path, max_pages: int) -> List[str]:
    """Defensive PDF reader. Raises exceptions to caller for fail-fast per-file handling."""
    with pdf_path.open("rb") as f:
        reader = pypdf.PdfReader(f)
        pages_to_scan = min(max_pages, len(reader.pages))
        return [(reader.pages[i].extract_text() or "") for i in range(pages_to_scan)]

In [35]:
### ==== EXTRACTION ENGINE (PURE FUNCTIONS) ====
@dataclass
class ExtractionHit:
    cleaned_name: str
    raw_fragment: str
    page: int
    confidence: float
    reason: str


def standardize_text(text: str) -> str:
    """Normalize OCR artifacts, ligatures, dashes, and whitespace."""
    ligatures = {
        "\ufb00": "ff",
        "\ufb01": "fi",
        "\ufb02": "fl",
        "\u2013": "-",
        "\u2014": "-",
        "\u00a0": " ",
    }
    normalized = text
    for src, dst in ligatures.items():
        normalized = normalized.replace(src, dst)
    normalized = re.sub(r"[ \t]+", " ", normalized)
    normalized = re.sub(r"\n{3,}", "\n\n", normalized)
    return normalized.strip()


def strip_leading_labels_and_titles(text: str) -> str:
    """Remove leading supervisor labels and academic titles from a fragment."""
    value = text
    value = re.sub(
        r"(?i)^\s*(?:supervisors?|main\s+supervisor|co-?supervisor|advisors?|advisers?|vejledere?|supervised by)\s*[:\-]?\s*",
        "",
        value,
    )
    value = re.sub(
        r"(?i)\b(?:associate\s+professor|assistant\s+professor|professor|prof\.?|ph\.?d\.?|msc|bsc)\b",
        "",
        value,
    )
    value = re.sub(r"\s+", " ", value)
    return value.strip(" ,.-:;\t")


def split_around_related_exclusion_terms(candidate: str, terms: Sequence[str]) -> List[str]:
    """
    Cut out exclusion terms and evaluate text before/after as potential candidates.
    Example: "John Doe, PhD" -> ["John Doe"]
    """
    working = [candidate]
    for term in sorted(set(terms), key=len, reverse=True):
        next_working: List[str] = []
        for part in working:
            if re.search(re.escape(term), part, flags=re.IGNORECASE):
                split_parts = re.split(re.escape(term), part, flags=re.IGNORECASE)
                next_working.extend([p.strip(" ,.-:;\t") for p in split_parts if p.strip(" ,.-:;\t")])
            else:
                next_working.append(part)
        working = next_working
    return [w for w in working if w]


def find_supervisor_contexts(pages: Sequence[str], keywords: Sequence[str]) -> List[Tuple[int, str, str]]:
    """Return (page_number, context_line, full_page_text) for likely supervisor contexts."""
    contexts: List[Tuple[int, str, str]] = []
    keyword_pattern = re.compile(r"|".join(re.escape(k) for k in keywords), flags=re.IGNORECASE)

    for idx, page_text in enumerate(pages, start=1):
        cleaned_page = standardize_text(page_text)
        lines = [ln.strip() for ln in cleaned_page.splitlines() if ln.strip()]

        for line in lines:
            if keyword_pattern.search(line):
                contexts.append((idx, line, cleaned_page))

    return contexts


def clean_candidate_name(candidate: str) -> str:
    candidate = strip_leading_labels_and_titles(candidate)
    candidate = re.sub(r"\s+", " ", candidate)
    return candidate.strip(" ,.-:;\t")


def canonicalize_name(name: str) -> str:
    lowered = name.lower().strip()
    lowered = re.sub(r"[^a-zæøå\s\-']", "", lowered)
    return re.sub(r"\s+", " ", lowered).strip()


def validate_name_format(name: str, config: Dict[str, object]) -> bool:
    words = [w for w in name.split() if w]
    if len(words) < 2 or len(words) > 6:
        return False

    name_lower = name.lower()
    not_name_phrases = config["NOT_NAME_PHRASES"]  # type: ignore[index]
    role_terms = config["ROLE_TERMS"]  # type: ignore[index]

    if any(phrase in name_lower for phrase in not_name_phrases):
        return False
    if any(term in name_lower for term in role_terms):
        return False
    if ":" in name:
        return False

    return True


def score_candidate(name: str, raw_fragment: str, used_nlp_person: bool, config: Dict[str, object]) -> float:
    th = config["CONFIDENCE_THRESHOLDS"]  # type: ignore[index]

    score = 0.0
    if KEYWORD_LINE_REGEX.search(raw_fragment):
        score += th["keyword_bonus"]  # type: ignore[index]
    if NAME_REGEX.search(name):
        score += th["regex_bonus"]  # type: ignore[index]
    if used_nlp_person:
        score += th["nlp_person_bonus"]  # type: ignore[index]

    tokens = name.split()
    if 2 <= len(tokens) <= 4:
        score += th["length_bonus"]  # type: ignore[index]

    lower_name = name.lower()
    if any(term in lower_name for term in config["ROLE_TERMS"]):  # type: ignore[index]
        score -= th["penalty_role_term"]  # type: ignore[index]

    return max(0.0, min(1.0, score))


def extract_names_from_context_line(line: str, config: Dict[str, object]) -> List[Tuple[str, str, bool]]:
    """Return tuples of (cleaned_candidate, raw_fragment, used_nlp_person)."""
    results: List[Tuple[str, str, bool]] = []

    # First try label-based extraction.
    m = KEYWORD_LINE_REGEX.search(line)
    raw_right_side = m.group(1).strip() if m else line

    # Split broad compound candidate line.
    split_parts = [p.strip() for p in SPLIT_REGEX.split(raw_right_side) if p.strip()]

    for part in split_parts:
        exclusion_parts = split_around_related_exclusion_terms(
            part,
            config["RELATED_EXCLUSION_TERMS"],  # type: ignore[index]
        )

        for ex_part in exclusion_parts:
            cleaned_fragment = clean_candidate_name(ex_part)
            if not cleaned_fragment:
                continue

            candidate_pool: List[Tuple[str, bool]] = []

            # NLP candidates
            if nlp is not None:
                doc = nlp(cleaned_fragment)
                for ent in doc.ents:
                    if ent.label_ == "PERSON":
                        candidate_pool.append((clean_candidate_name(ent.text), True))

            # Regex candidates (can recover names that NLP misses)
            for match in NAME_REGEX.finditer(cleaned_fragment):
                candidate_pool.append((clean_candidate_name(match.group(0)), False))

            # Always include fragment fallback to avoid losing names due partial NLP output.
            candidate_pool.append((cleaned_fragment, False))

            # Local dedupe
            seen_local = set()
            for candidate_name, used_nlp_person in candidate_pool:
                key = canonicalize_name(candidate_name)
                if key and key not in seen_local:
                    seen_local.add(key)
                    results.append((candidate_name, line, used_nlp_person))

    return results


def deduplicate_hits(hits: Sequence[ExtractionHit]) -> List[ExtractionHit]:
    """Keep the highest-confidence hit per canonicalized name."""
    best_by_name: Dict[str, ExtractionHit] = {}
    for hit in hits:
        key = canonicalize_name(hit.cleaned_name)
        if not key:
            continue
        existing = best_by_name.get(key)
        if existing is None or hit.confidence > existing.confidence:
            best_by_name[key] = hit
    return sorted(best_by_name.values(), key=lambda h: (-h.confidence, h.cleaned_name))


def extract_supervisors_from_pdf(pdf_path: Path, config: Dict[str, object]) -> List[ExtractionHit]:
    pages = load_pdf_pages(pdf_path, max_pages=config["MAX_PAGES_TO_SCAN"])  # type: ignore[index]
    contexts = find_supervisor_contexts(pages, config["EXTRACTION_KEYWORDS"])  # type: ignore[index]

    hits: List[ExtractionHit] = []
    min_conf = config["CONFIDENCE_THRESHOLDS"]["min_confidence"]  # type: ignore[index]

    for page, context_line, _ in contexts:
        extracted = extract_names_from_context_line(context_line, config)
        for candidate_name, raw_fragment, used_nlp_person in extracted:
            cleaned = clean_candidate_name(candidate_name)
            if not validate_name_format(cleaned, config):
                continue

            confidence = score_candidate(cleaned, raw_fragment, used_nlp_person, config)
            if confidence < min_conf:
                continue

            hits.append(
                ExtractionHit(
                    cleaned_name=cleaned,
                    raw_fragment=raw_fragment,
                    page=page,
                    confidence=confidence,
                    reason="keyword-context",
                )
            )

    return deduplicate_hits(hits)

In [36]:
### ==== EXECUTION & DEBUG LOOP ====
pdf_files = list_local_pdf_files(
    local_dir=CONFIG["LOCAL_DATA_DIR"],  # type: ignore[index]
    max_files=CONFIG["MAX_FILES"],  # type: ignore[index]
)

results: List[Dict[str, object]] = []
failures: List[Dict[str, str]] = []

for i, pdf_path in enumerate(pdf_files, start=1):
    logger.info("[%d/%d] Processing %s", i, len(pdf_files), pdf_path.name)

    try:
        hits = extract_supervisors_from_pdf(pdf_path, CONFIG)

        if not hits:
            logger.info("No supervisor hits above confidence threshold for %s", pdf_path.name)

        for hit in hits:
            # Side-by-side debug output: raw extraction context vs cleaned candidate.
            print(
                f"{pdf_path.name} | p.{hit.page} | conf={hit.confidence:.2f}\n"
                f"  raw   : {hit.raw_fragment}\n"
                f"  clean : {hit.cleaned_name}\n"
            )

            results.append(
                {
                    "pdf_file": pdf_path.name,
                    "page": hit.page,
                    "raw_fragment": hit.raw_fragment,
                    "cleaned_name": hit.cleaned_name,
                    "confidence": hit.confidence,
                    "reason": hit.reason,
                }
            )

    except Exception as exc:
        logger.exception("Failed processing %s", pdf_path.name)
        failures.append({"pdf_file": pdf_path.name, "error": str(exc)})
        continue

print("\n=== BATCH SUMMARY ===")
print(f"Processed files : {len(pdf_files)}")
print(f"Extracted hits  : {len(results)}")
print(f"Failed files    : {len(failures)}")

if failures:
    print("\nFailed files detail:")
    for failure in failures:
        print(f"- {failure['pdf_file']}: {failure['error']}")

INFO | PDF discovery complete | directory=/Users/oliver/Desktop/MSc_Speciale/Thesis_OCR/Data/RAW_test | files=23
INFO | [1/23] Processing 5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf
INFO | [2/23] Processing 5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf
INFO | [3/23] Processing 5d04d261d9001d010a45c03d_Comparativ analysis of intervention strategies for a high rise building using Fire Brigade Intervention Model indsatsstr.pdf


5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf | p.5 | conf=0.55
  raw   : I would like to thank my supervisor at DTU, Ole Ravn, for always supporting me whilst I was
  clean : Ole Ravn

5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf | p.2 | conf=0.55
  raw   : Vejleder Pawel Wargocki
  clean : Pawel Wargocki



INFO | [4/23] Processing 5d1c8d6bd9001d146569a4b3_Characterising eye-motion patterns in patients with autistic spectrum disorders using machine learning (translated Karak.pdf
INFO | [5/23] Processing 5d1c8d79d9001d146569a4dc_Deep speech recognition in Danish (translated Dyb talegenkendelse pa dansk).pdf
WARNING | Ignoring wrong pointing object 366 0 (offset 0)
WARNING | Ignoring wrong pointing object 438 0 (offset 0)
WARNING | Ignoring wrong pointing object 465 0 (offset 0)
WARNING | Ignoring wrong pointing object 617 0 (offset 0)
WARNING | Ignoring wrong pointing object 865 0 (offset 0)
WARNING | Ignoring wrong pointing object 1055 0 (offset 0)


5d04d261d9001d010a45c03d_Comparativ analysis of intervention strategies for a high rise building using Fire Brigade Intervention Model indsatsstr.pdf | p.3 | conf=0.55
  raw   : Supervisor Associate professor - DTU: Anne Simone Dederichs, Ph. D.
  clean : Anne Simone

5d04d261d9001d010a45c03d_Comparativ analysis of intervention strategies for a high rise building using Fire Brigade Intervention Model indsatsstr.pdf | p.1 | conf=0.55
  raw   : Supervisor Associate professor Anne Simone Dederichs, Ph. D.
  clean : Anne Simone Dederichs

5d1c8d6bd9001d146569a4b3_Characterising eye-motion patterns in patients with autistic spectrum disorders using machine learning (translated Karak.pdf | p.1 | conf=0.90
  raw   : Supervisors: Morten Mørup, Paolo Masulli and Tobias Andersen
  clean : Paolo Masulli

5d1c8d6bd9001d146569a4b3_Characterising eye-motion patterns in patients with autistic spectrum disorders using machine learning (translated Karak.pdf | p.1 | conf=0.90
  raw   : Supervisors: Mort

INFO | No supervisor hits above confidence threshold for 5d1c8d79d9001d146569a4dc_Deep speech recognition in Danish (translated Dyb talegenkendelse pa dansk).pdf
INFO | [6/23] Processing 5d1c8d7bd9001d146569a4ec_Impact of probiotic tropodithietic acid (TDA) producing Phaeobacter piscinae on microbial community members related to a.pdf
INFO | No supervisor hits above confidence threshold for 5d1c8d7bd9001d146569a4ec_Impact of probiotic tropodithietic acid (TDA) producing Phaeobacter piscinae on microbial community members related to a.pdf
INFO | [7/23] Processing 5d1c8d83d9001d146569a4fd_Storage of heat in the pipeline of existing district heating grid with heat pump supply (translated Lagring af varme i r.pdf
INFO | No supervisor hits above confidence threshold for 5d1c8d83d9001d146569a4fd_Storage of heat in the pipeline of existing district heating grid with heat pump supply (translated Lagring af varme i r.pdf
INFO | [8/23] Processing 5d1c8d92d9001d146569a522_Investigating the impact

...

68841cc6439dc00102b37cac_Assessment of solar energy in Greenland Analyzing potential and resource availability (translated Analyse af potentialet.pdf | p.4 | conf=0.55
  raw   : I would like to express my sincere gratitude to my supervisor, Adam Rasmus Jensen, and my co-
  clean : Adam Rasmus Jensen

68841cc6439dc00102b37cac_Assessment of solar energy in Greenland Analyzing potential and resource availability (translated Analyse af potentialet.pdf | p.4 | conf=0.55
  raw   : supervisors, Ioannis Sifnaios and Janne Dragsted, for providing invaluable guidance, feedback, and
  clean : Janne Dragsted



INFO | [22/23] Processing dtu_findit-master_thesis-5cf10bded9001d206429933c_Restoration of marine boulder reefs addressing the single large or several small (SLOSS) debate in relation to marine ha.pdf


TEST.pdf | p.1 | conf=0.65
  raw   : Supervisor: Jon C. Svendsen, Senior researcher, DTU aqua
  clean : Jon C. Svendsen



INFO | [23/23] Processing dtu_findit-master_thesis-5d3d82e9d9001d32f558c087_Spatial variation and uncertainty in contaminant characterization of clayey tills (translated Rumlig variation og usikke.pdf
INFO | No supervisor hits above confidence threshold for dtu_findit-master_thesis-5d3d82e9d9001d32f558c087_Spatial variation and uncertainty in contaminant characterization of clayey tills (translated Rumlig variation og usikke.pdf


dtu_findit-master_thesis-5cf10bded9001d206429933c_Restoration of marine boulder reefs addressing the single large or several small (SLOSS) debate in relation to marine ha.pdf | p.1 | conf=0.65
  raw   : Supervisor: Jon C. Svendsen, Senior researcher, DTU aqua
  clean : Jon C. Svendsen


=== BATCH SUMMARY ===
Processed files : 23
Extracted hits  : 17
Failed files    : 0


In [37]:
for pdf_file in set(r["pdf_file"] for r in results):
    print(f"Results for {pdf_file}:")
    for item in results:
        if item["pdf_file"] == pdf_file:
            print(f"  Page {item['page']} | Conf {item['confidence']:.2f} | Cleaned: {item['cleaned_name']}")
    print()



Results for 5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf:
  Page 2 | Conf 0.55 | Cleaned: Pawel Wargocki

Results for dtu_findit-master_thesis-5cf10bded9001d206429933c_Restoration of marine boulder reefs addressing the single large or several small (SLOSS) debate in relation to marine ha.pdf:
  Page 1 | Conf 0.65 | Cleaned: Jon C. Svendsen

Results for 5d04d261d9001d010a45c03d_Comparativ analysis of intervention strategies for a high rise building using Fire Brigade Intervention Model indsatsstr.pdf:
  Page 3 | Conf 0.55 | Cleaned: Anne Simone
  Page 1 | Conf 0.55 | Cleaned: Anne Simone Dederichs

Results for 68841cc6439dc00102b37cac_Assessment of solar energy in Greenland Analyzing potential and resource availability (translated Analyse af potentialet.pdf:
  Page 4 | Conf 0.55 | Cleaned: Adam Rasmus Jensen
  Page 4 | Conf 0.55 | Cleaned: Janne Dragsted

Results for 68841cc3762f9c010279